# CYP1A2 inhibitor classification: k-NN, random forest and SVM

The training and test sets come from `ShuffleandSplit` and are the same splits used for the MOE Binary QSAR models. A compound with an Activity Score of 40 or above is an inhibitor (1); everything below 40, including inconclusive compounds, is a non-inhibitor (0).

In [ ]:
%matplotlib inline
from cyp_models import (
    CLASSIFIERS, evaluate, load_split, plot_calibration, rf_calibration, roc_test,
)

In [ ]:
X_train, y_train, X_test, y_test = load_split("1a2")
print(f"Training: {len(y_train)} compounds, {y_train.sum()} inhibitors, "
      f"{len(y_train) - y_train.sum()} non-inhibitors")
print(f"Test:     {len(y_test)} compounds, {y_test.sum()} inhibitors, "
      f"{len(y_test) - y_test.sum()} non-inhibitors")
print(f"Features: {X_train.shape[1]} MOE 2-D descriptors")

## Classifiers

Hyperparameters match the scikit-learn 0.15 defaults used in the original analysis.

In [ ]:
CLASSIFIERS

## Validation

- **Training set:** 5-fold cross-validation.
- **Test set:** each model is fit on the full training set and then predicts the held-out test set. The scaler is part of the pipeline, so it too is fit on training data only.

In [ ]:
results, test_confusion = evaluate(X_train, y_train, X_test, y_test)
results

## Test set confusion matrices

Rows are actual classes and columns are predicted classes, in the order non-inhibitor, inhibitor.

In [ ]:
for name, cm in test_confusion.items():
    print(name)
    print(cm, end='\n\n')

## Random forest calibration

Each point is one predicted probability (a multiple of 0.1, since the forest has 10 trees). Its height is the fraction of compounds given that probability that are actually inhibitors, and its area is proportional to the number of such compounds. Points on the diagonal are perfectly calibrated.

In [ ]:
calibration = rf_calibration(X_train, y_train, X_test, y_test)
fig = plot_calibration(calibration, baseline=y_train.mean(), title="CYP1A2: random forest calibration")
fig.savefig("results/1a2_rf_calibration.png", dpi=150)
calibration["Test"]

## ROC curves on the test set

Each model is fit on the training set. Random forest and k-NN are scored by predicted probability, the SVM by its decision function.

In [ ]:
auc, fig = roc_test(X_train, y_train, X_test, y_test, title="CYP1A2: test set ROC")
fig.savefig("results/1a2_roc.png", dpi=150)
auc

In [ ]:
results.merge(auc.assign(Set="Test"), how="left").to_csv("results/1a2.csv", index=False)